In [1]:
pip install pyrealsense2

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pyrealsense2 as rs
import cv2
import numpy as np
import os
import re

class realsense_camera:
    is_opened = False
    config = None
    intr = None

    def __init__(self, height=480, width=640, fps=30, use_color=True):
        self.height = height
        self.width = width
        self.fps = fps
        self.use_color = use_color

        # RealSense 파이프라인 & 설정
        pipeline = rs.pipeline()
        config = rs.config()

        pipeline_wrapper = rs.pipeline_wrapper(pipeline)
        self.config = config
        self.pipeline = pipeline
        self.pipeline_wrapper = pipeline_wrapper

        # 디바이스 연결 체크
        if self.can_connect():
            pipeline_profile = config.resolve(pipeline_wrapper)
            device = pipeline_profile.get_device()

            # RGB 카메라 있는지 체크
            found_rgb = False
            for s in device.sensors:
                if s.get_info(rs.camera_info.name) == 'RGB Camera':
                    found_rgb = True

            if not found_rgb:
                self.use_color = False

            # RGB 스트림 설정
            if self.use_color:
                config.enable_stream(rs.stream.color, width, height, rs.format.bgr8, fps)

            # 파이프라인 시작
            if self.can_connect() and self.use_color:
                self.is_opened = True
                pipeline.start(config)
                self.intr = pipeline.get_active_profile().get_stream(
                    rs.stream.color
                ).as_video_stream_profile().get_intrinsics()

    def get_intrinsics(self):
        return self.intr

    def set(self, num, value):
        # 해상도 / FPS 변경
        if num == cv2.CAP_PROP_FRAME_HEIGHT:
            self.height = int(value)
        elif num == cv2.CAP_PROP_FRAME_WIDTH:
            self.width = int(value)
        elif num == cv2.CAP_PROP_FPS:
            self.fps = int(value)

        if self.is_opened:
            # 스트림 재설정
            self.config.disable_all_streams()
            if self.use_color:
                self.config.enable_stream(
                    rs.stream.color,
                    self.width,
                    self.height,
                    rs.format.bgr8,
                    self.fps
                )
            if self.can_connect():
                self.pipeline.stop()
                self.pipeline.start(self.config)
                if self.use_color:
                    self.intr = self.pipeline.get_active_profile().get_stream(
                        rs.stream.color
                    ).as_video_stream_profile().get_intrinsics()

    def get(self, num):
        if num == cv2.CAP_PROP_FRAME_HEIGHT:
            return self.height
        elif num == cv2.CAP_PROP_WIDTH:
            return self.width
        elif num == cv2.CAP_PROP_FPS:
            return self.fps

    def can_connect(self):
        # config의 유효성 및 카메라 연결 여부 체크
        try:
            return self.config.can_resolve(self.pipeline_wrapper)
        except Exception:
            return False

    def isOpened(self):
        return self.is_opened

    def release(self):
        if self.pipeline is not None:
            if self.isOpened():
                if self.can_connect():
                    self.pipeline.stop()

    def read(self):
        """컬러 프레임만 읽기"""
        try:
            frames = self.pipeline.wait_for_frames(100)
            if self.use_color:
                color_frame = frames.get_color_frame()
                if not color_frame:
                    return False, None
                color_image = np.asanyarray(color_frame.get_data())
            else:
                color_image = np.zeros((self.height, self.width, 3), dtype=np.uint8)
            return True, color_image
        except Exception:
            return False, None


def get_next_save_index(ext=".png", search_dir="."):
    """
    search_dir 안에 있는 000.png, 001.png ... 파일 중
    가장 큰 번호를 찾고, 그 다음 번호를 리턴.
    파일이 없으면 0부터 시작.
    """
    pattern = re.compile(rf"^(\d+){re.escape(ext)}$")
    max_idx = -1

    try:
        for fname in os.listdir(search_dir):
            match = pattern.match(fname)
            if match:
                idx = int(match.group(1))
                if idx > max_idx:
                    max_idx = idx
    except FileNotFoundError:
        pass

    return max_idx + 1  # 없으면 0, 있으면 max+1


if __name__ == "__main__":

    # ===== 저장 폴더 설정 =====
    category = 'cube'
    save_category = f"./data/{category}/train/good"      # 나중에 "anomaly" 등으로 변경
    #save_category = f"./data/{category}/test/good"
    #save_category = f"./data/{category}/test/anomaly"
    save_dir = save_category
    os.makedirs(save_dir, exist_ok=True)
    # =========================

    # ===== 센터 크롭 사이즈 (정사각형) =====
    # 예: 512 -> 512x512 센터 크롭
    crop_size = 512
    # =====================================

    cam = realsense_camera(use_color=True)
    # 해상도 / FPS 설정
    cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    cam.set(cv2.CAP_PROP_FPS, 30)

    if cam.isOpened():
        # 해당 폴더 안에서 이어서 번호 붙이기
        save_idx = get_next_save_index(ext=".png", search_dir=save_dir)
        print(f"Save dir: {save_dir}, start index: {save_idx}")

        while True:
            ret, frame = cam.read()
            if not ret or frame is None:
                continue

            # ===== 센터 정사각형 크롭 =====
            h, w = frame.shape[:2]
            # crop_size가 이미지보다 크면 자동으로 줄이기
            size = min(crop_size, h, w)

            x1 = (w - size) // 2
            y1 = (h - size) // 2
            x2 = x1 + size
            y2 = y1 + size

            frame_cropped = frame[y1:y2, x1:x2]
            # =================================

            cv2.imshow("color", frame_cropped)

            key = cv2.waitKey(1) & 0xFF

            # q → 종료
            if key == ord('q'):
                break

            # s → 크롭된 이미지 저장
            elif key == ord('s'):
                filename = os.path.join(save_dir, f"{save_idx:03d}.png")
                cv2.imwrite(filename, frame_cropped)
                print("Saved:", filename)
                save_idx += 1

    cam.release()
    cv2.destroyAllWindows()